# 01 — Exploratory Data Analysis

**Hospital Readmission Risk Predictor**

Before diving into modelling, we need to deeply understand the data we are working with.
This notebook walks through the Diabetes 130-US Hospitals dataset, examining class distributions,
missing values, demographic patterns, and clinical signals. The goal is to build intuition about
what drives 30-day readmissions so we can make informed feature engineering and modelling decisions.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_and_prepare
from src.utils import set_plot_style, PALETTE

set_plot_style()
pd.set_option('display.max_columns', 50)
print('Setup complete ✓')

## 1. Load the Dataset

We start by loading the raw data and encoding our binary target. The original `readmitted` column
has three values: `<30`, `>30`, and `NO`. For this project, we focus on **30-day readmission**,
so `<30` → 1 and everything else → 0.

In [ ]:
df = load_and_prepare()
print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Target Variable — Class Imbalance Analysis

One of the first things we need to understand is how rare our positive class is. In healthcare,
readmission within 30 days is thankfully uncommon — but that means our model will be trained on
a heavily imbalanced dataset, which has major implications for our choice of metrics and training
strategy.

In [ ]:
target_counts = df['readmit_30d'].value_counts()
target_pct = df['readmit_30d'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
axes[0].bar(['Not Readmitted (0)', 'Readmitted <30d (1)'],
            target_counts.values, color=[PALETTE[0], PALETTE[3]], edgecolor='white')
axes[0].set_ylabel('Count')
axes[0].set_title('Target Class Distribution')
for i, (cnt, pct) in enumerate(zip(target_counts.values, target_pct.values)):
    axes[0].text(i, cnt + 500, f'{cnt:,}\n({pct:.1f}%)', ha='center', fontsize=11)

# Pie chart
axes[1].pie(target_counts.values, labels=['Not Readmitted', 'Readmitted <30d'],
            colors=[PALETTE[0], PALETTE[3]], autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 12})
axes[1].set_title('Readmission Rate')

fig.suptitle('30-Day Readmission: Class Imbalance', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f'Imbalance ratio: {target_counts[0]/target_counts[1]:.1f}:1')

### Findings

The positive class (readmitted within 30 days) makes up roughly **11%** of the dataset. This ~9:1
imbalance means:

- **Accuracy is misleading** — a model that predicts "not readmitted" for everyone achieves ~89% accuracy.
- We must use **precision-recall AUC** and **recall** as primary metrics.
- We will need **class weighting** or **resampling** strategies during training.

## 3. Missing Values Analysis

Healthcare datasets are notoriously messy. In this dataset, missing values are encoded as `?`
(which we've already converted to NaN). Let's understand the missingness patterns.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'percent': missing_pct})
missing_df = missing_df[missing_df['count'] > 0].sort_values('percent', ascending=False)

print(f'Columns with missing values: {len(missing_df)}')
display(missing_df)

In [ ]:
# Missing value heatmap for columns with >1% missing
cols_with_missing = missing_df[missing_df['percent'] > 1].index.tolist()

if cols_with_missing:
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(x=missing_df.index[:15], y=missing_df['percent'][:15],
                palette='Reds_r', ax=ax)
    ax.set_ylabel('Missing (%)')
    ax.set_title('Missing Values by Column (Top 15)')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

### Findings

- **weight** is missing in ~97% of records — virtually useless; we'll drop it.
- **payer_code** (~40% missing) and **medical_specialty** (~49% missing) are too sparse to impute reliably.
- **race** (~2% missing) — we'll create an explicit `Unknown` category rather than imputing with mode,
  because imputing with the majority race would introduce bias.
- **diag_1/2/3** have minimal missingness and will be mapped to clinical categories.

## 4. Readmission by Demographics

Understanding how readmission rates vary across demographic groups is critical for both
model development and fairness auditing.

In [ ]:
# Readmission by Race
race_readmit = df.groupby('race')['readmit_30d'].agg(['mean', 'count']).reset_index()
race_readmit.columns = ['race', 'readmit_rate', 'count']
race_readmit = race_readmit.sort_values('count', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=race_readmit, x='race', y='readmit_rate', palette=PALETTE, ax=axes[0])
axes[0].set_title('30-Day Readmission Rate by Race')
axes[0].set_ylabel('Readmission Rate')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')

sns.barplot(data=race_readmit, x='race', y='count', palette=PALETTE, ax=axes[1])
axes[1].set_title('Sample Size by Race')
axes[1].set_ylabel('Number of Encounters')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()
display(race_readmit)

In [ ]:
# Readmission by Gender
gender_readmit = df.groupby('gender')['readmit_30d'].agg(['mean', 'count']).reset_index()
gender_readmit.columns = ['gender', 'readmit_rate', 'count']

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=gender_readmit, x='gender', y='readmit_rate',
            palette=[PALETTE[0], PALETTE[1]], ax=ax)
ax.set_title('30-Day Readmission Rate by Gender')
ax.set_ylabel('Readmission Rate')
plt.tight_layout()
plt.show()
display(gender_readmit)

In [ ]:
# Readmission by Age
age_readmit = df.groupby('age')['readmit_30d'].agg(['mean', 'count']).reset_index()
age_readmit.columns = ['age', 'readmit_rate', 'count']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=age_readmit, x='age', y='readmit_rate', palette='viridis', ax=axes[0])
axes[0].set_title('30-Day Readmission Rate by Age Group')
axes[0].set_ylabel('Readmission Rate')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

sns.barplot(data=age_readmit, x='age', y='count', palette='viridis', ax=axes[1])
axes[1].set_title('Sample Size by Age Group')
axes[1].set_ylabel('Count')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

### Findings

- **Race:** Readmission rates are fairly similar across racial groups (within 1–2 percentage points),
  but the dataset is heavily skewed towards Caucasian patients (~75%). This imbalance means model
  performance may be less reliable for minority groups.
- **Gender:** Male and female readmission rates are comparable — gender alone is not a strong signal.
- **Age:** Older patients (60–90) have slightly higher readmission rates. The bulk of encounters
  are in the 60-80 range, which is expected for a diabetic population.

## 5. Clinical Feature Analysis

In [ ]:
# Key numeric features vs readmission
numeric_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
                'num_medications', 'number_outpatient', 'number_emergency',
                'number_inpatient', 'number_diagnoses']

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
for ax, col in zip(axes.ravel(), numeric_cols):
    df.groupby('readmit_30d')[col].mean().plot(kind='bar', ax=ax,
        color=[PALETTE[0], PALETTE[3]], edgecolor='white')
    ax.set_title(col, fontsize=11)
    ax.set_xticklabels(['No', 'Yes'], rotation=0)
    ax.set_xlabel('')

fig.suptitle('Mean Feature Value by Readmission Status', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix for numeric features
corr_cols = numeric_cols + ['readmit_30d']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

### Findings

- **number_inpatient** shows the clearest difference between readmitted and non-readmitted patients.
  Prior inpatient visits are a well-established risk factor in the literature.
- **number_emergency** and **number_diagnoses** also show meaningful signal.
- Correlations with the target are weak (typical for tabular healthcare data), suggesting that
  non-linear models will be needed to capture interactions.

## 6. Medication Analysis

In [ ]:
# Medication change vs readmission
med_change = df.groupby('change')['readmit_30d'].mean()
diabetes_med = df.groupby('diabetesMed')['readmit_30d'].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

med_change.plot(kind='bar', ax=axes[0], color=[PALETTE[0], PALETTE[2]], edgecolor='white')
axes[0].set_title('Readmission Rate by Medication Change')
axes[0].set_ylabel('Readmission Rate')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

diabetes_med.plot(kind='bar', ax=axes[1], color=[PALETTE[0], PALETTE[2]], edgecolor='white')
axes[1].set_title('Readmission Rate by Diabetes Medication Use')
axes[1].set_ylabel('Readmission Rate')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Insulin patterns
insulin_readmit = df.groupby('insulin')['readmit_30d'].agg(['mean', 'count']).reset_index()
insulin_readmit.columns = ['insulin', 'readmit_rate', 'count']

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=insulin_readmit, x='insulin', y='readmit_rate', palette=PALETTE, ax=ax)
ax.set_title('Readmission Rate by Insulin Status')
ax.set_ylabel('Readmission Rate')
plt.tight_layout()
plt.show()
display(insulin_readmit)

### Findings

- Patients whose medications were **changed** during the encounter have a slightly *higher* readmission
  rate — this likely reflects clinical instability rather than a causal effect of the medication change.
- Patients on **insulin** (especially those whose dose was adjusted) show somewhat different readmission
  patterns, which aligns with the clinical intuition that insulin-dependent diabetics tend to be sicker.

## 7. Discharge Disposition & Leakage Risk

In [ ]:
# Discharge disposition analysis
disp_readmit = df.groupby('discharge_disposition_id')['readmit_30d'].agg(
    ['mean', 'count']).reset_index()
disp_readmit.columns = ['discharge_id', 'readmit_rate', 'count']
disp_readmit = disp_readmit.sort_values('count', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=disp_readmit, x='discharge_id', y='readmit_rate', palette='coolwarm', ax=ax)
ax.set_title('Readmission Rate by Discharge Disposition (Top 15 by Volume)')
ax.set_ylabel('Readmission Rate')
ax.set_xlabel('Discharge Disposition ID')
plt.tight_layout()
plt.show()

# Flag leakage codes
leakage_ids = [11, 13, 14, 19, 20, 21]
leakage_count = df['discharge_disposition_id'].isin(leakage_ids).sum()
print(f'\nRows with leakage discharge codes (expired/hospice): {leakage_count:,}')
print(f'These patients cannot be readmitted and will be removed during preprocessing.')

## 8. Clinical Summary

After this exploratory analysis, here are the key takeaways that will guide our modelling:

| Finding | Implication |
|---------|-------------|
| ~11% positive rate (heavily imbalanced) | Use class weights, evaluate with PR-AUC and recall |
| `weight` column 97% missing | Drop it entirely |
| `race` 2% missing | Fill with `Unknown` (fairness-preserving) |
| Prior inpatient visits are strongest signal | Create aggregate utilisation features |
| Medication changes correlate with readmission | Create medication-change count features |
| Discharge codes 11/13/14/19/20/21 are leakage | Remove expired/hospice patients |
| Weak linear correlations with target | Non-linear tree models likely needed |
| Demographic subgroups are imbalanced | Fairness audit required post-modelling |

With these insights, we're ready to move to feature engineering and modelling.